# Saathi — QLoRA fine-tune on a free T4

Trains the answer writer that sits behind Saathi's citation guard.

**What this adapter does:** learns the output contract — read numbered statutory extracts, return JSON that
cites them by index, hedge when the context does not answer. **What it cannot do:** invent law. The corpus
in `src/lib/corpus/` stays the only source of legal content, and the guard drops anything uncited.

**Runtime → Change runtime type → T4 GPU.** CPU will not work; the script refuses rather than silently
training in fp32. Expect roughly 10–15 minutes end to end.

The dataset is committed at `data/lora/{train,eval}.jsonl`, so no Node toolchain is needed here. To
regenerate it from the corpus after editing provisions, run `npm run dataset` in the repo.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| bf16 supported', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'

In [ ]:
import os, subprocess
REPO = '/content/Saathi'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/lightning2661/Saathi.git', REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements-colab.txt
import transformers, peft, trl, datasets
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| trl', trl.__version__, '| datasets', datasets.__version__)

In [ ]:
import json, pathlib
for name in ('train', 'eval'):
    p = pathlib.Path('data/lora') / f'{name}.jsonl'
    rows = [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
    kinds = {}
    for r in rows:
        k = r.get('meta', {}).get('kind', 'grounded')
        kinds[k] = kinds.get(k, 0) + 1
    print(f'{name}: {len(rows)} rows', kinds)

sample = json.loads(pathlib.Path('data/lora/train.jsonl').read_text(encoding='utf-8').splitlines()[0])
print('\n--- user turn (truncated) ---\n', sample['messages'][1]['content'][:600])
print('\n--- target ---\n', sample['messages'][2]['content'][:400])

In [ ]:
# Qwen2.5-1.5B-Instruct is ungated, which matters on a free account.
# Any instruct checkpoint with a chat template works; pass --base-model.
!python scripts/train_lora.py \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --epochs 3 \
  --rank 16 \
  --alpha 32 \
  --max-seq-len 2048 \
  --out lora-out/saathi-lora

In [ ]:
# Same scoring code as CI. In-process generation keeps this off vLLM.
!python scripts/eval_lora.py \
  --backend transformers \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --adapter lora-out/saathi-lora \
  --json-out lora-out/eval.json

In [ ]:
import json
train = json.load(open('lora-out/train_report.json'))
ev = json.load(open('lora-out/eval.json'))

rows = [
    ('base model', train['base_model']),
    ('gpu', train['gpu']),
    ('precision', train['precision']),
    ('adapter rank', train['lora']['r']),
    ('train / eval rows', f"{train['rows']['train']} / {train['rows']['eval']}"),
    ('optimiser steps', train['steps']),
    ('train loss', round(train['train_loss'] or 0, 4)),
    ('eval loss', round(train['eval_loss'] or 0, 4)),
    ('', ''),
    ('schema ok', ev['schema_ok']),
    ('cite precision', ev['cite_precision']),
    ('cite recall', ev['cite_recall']),
    ('grounded rate', ev['grounded_rate']),
    ('invented section rate', ev['invented_section_rate']),
    ('abstain accuracy', ev['abstain_accuracy']),
]
width = max(len(str(k)) for k, _ in rows) + 2
print('SAATHI ADAPTER SCORECARD')
print('=' * (width + 24))
for k, v in rows:
    print(f'{str(k):<{width}}{v}')
print('=' * (width + 24))
print('cite_precision and invented_section_rate are the ones that matter:')
print('the guard will drop what fails them, so a low score means silent fallbacks, not bad advice.')

### Reading the numbers

- **schema_ok** below 1.0 means the model is drifting into prose. Raise epochs or lower the learning rate.
- **cite_precision** is the share of citations that point at a real context item. Anything under ~0.95 means
  the guard will be discarding rights, and users will silently get the extractive answer instead.
- **invented_section_rate** must be near 0. A non-zero value is the failure this project exists to prevent.
- **cite_recall** compares against the corpus-authored answer, so 1.0 means the adapter picked the same
  provisions a legal reviewer would have picked from the same context.
- **eval loss** on 10 held-out rows is a weak signal. Treat the behavioural metrics as the real result.

### Using the adapter

```bash
# on a GPU host
bash scripts/serve_lora.sh lora-out/saathi-lora Qwen/Qwen2.5-1.5B-Instruct
export SAATHI_MODEL_PROVIDER=lora
export SAATHI_LORA_BASE_URL=http://127.0.0.1:8000/v1
```

To expose it publicly, `PUBLISH=1 bash scripts/serve_lora.sh` puts the token-checking, rate-limited proxy in
front of vLLM. An open GPU port is someone else's bill.

Saathi works without any of this: `SAATHI_MODEL_PROVIDER=extractive` is the default and calls no model.

In [ ]:
import shutil
shutil.make_archive('/content/saathi-lora', 'zip', 'lora-out/saathi-lora')
print('adapter archive ready: /content/saathi-lora.zip')

try:
    from google.colab import files
    files.download('/content/saathi-lora.zip')
except ImportError:
    pass